In [46]:
import pandas as pd
import numpy as np

# Giả sử 'df_raw' là dataframe chứa 31 thuộc tính ban đầu của bạn
# Load dataset

DATA_PATH = "../../data/final/scored_data.csv"

df_ready_for_kbrs = pd.read_csv(DATA_PATH)

In [47]:
filter_features = ['product_name', 'price_vnd', 'price_segment', 'weight_tier', 'score_perf', 'score_cam', 'score_batt', 'score_disp', 'ip_status']
df_ready_for_kbrs[filter_features].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
price_vnd,395.0,14136405.06,11796979.00,1590000.00,4740000.00,8990000.00,22690000.00,63990000.00
weight_tier,395.0,2.00,0.61,1.00,2.00,2.00,2.00,3.00
score_perf,364.0,3.89,1.92,0.02,2.41,3.42,5.06,8.99
score_cam,279.0,3.45,1.36,0.86,2.48,3.32,4.16,9.03
score_batt,303.0,4.46,1.42,0.91,3.33,4.33,5.22,8.00
score_disp,395.0,4.22,2.43,0.00,2.00,4.29,6.29,8.29
ip_status,395.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [48]:
PERF_THRESHOLD = 5.5 #Lấy mức lớn hơn 75%
CAM_THRESHOLD = 4.5

MAPPING_MATRIX_CONFIG = {
    'Gia_Re': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'price_segment': lambda x: x == 'budget'}
    },
    'Flagship': {
        'weights': {'perf': 0.3, 'cam': 0.3, 'batt': 0.1, 'disp': 0.3},
        'hard_filters': {'price_segment': lambda x: x == 'flagship'}
    },
    'Lien_Lac_Co_Ban': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {} # Không có điều kiện cứng
    },
    'Choi_Game': {
        'weights': {'perf': 0.7, 'cam': 0.05, 'batt': 0.15, 'disp': 0.1},
        'hard_filters': {
            'score_perf': lambda x: x >= PERF_THRESHOLD,      
            'refresh_rate_hz': lambda x: x >= 120.0, 
            'ram_gb': lambda x: x >= 8.0            
        }
    },
    'Giai_Tri_MXH': {
        'weights': {'perf': 0.2, 'cam': 0.1, 'batt': 0.3, 'disp': 0.4},
        'hard_filters': {'display_tier': lambda x: x >= 3}
    },
    'Chup_Hinh_Quay_Phim': {
        'weights': {'perf': 0.15, 'cam': 0.6, 'batt': 0.1, 'disp': 0.15},
        'hard_filters': {
            'score_cam': lambda x: x >= CAM_THRESHOLD,
            'video_resolution_rank': lambda x: x >= 4, #tu 4k tro len
            'storage_gb': lambda x: x >= 256.0
        }
    },
    'Pin_Trau_Sac_Nhanh': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {
            'battery_mah': lambda x: x >= 6000,
            'fast_charge_w': lambda x: x >= 33,
        }
    },

    'Nho_Gon_Nhe_Tay': {
        'weights': {'perf': 0.2, 'cam': 0.3, 'batt': 0.2, 'disp': 0.3},
        # ĐIỀU KIỆN CỨNG: Bắt buộc trọng lượng phải thuộc nhóm 1 (Dưới 185g)
        'hard_filters': {'weight_tier': lambda x: x == 1} 
    },

    'Khang_Nuoc': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'ip_certified': lambda x: x == 1}
    },

    'Bat_Ky': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {}
    }
}


In [49]:
# ==========================================
#  LÕI HỆ THỐNG KBRS (ENGINE)
# ==========================================
class KBRSEngine:
    def __init__(self, df_phones, mapping_matrix):
        self.df = df_phones.copy()
        self.mapping_matrix = mapping_matrix
        # Các cột điểm đã chuẩn hóa (0-10) dùng để nhân ma trận
        self.score_features = ['score_perf', 'score_cam', 'score_batt', 'score_disp']

    def recommend(self, user_needs, max_budget=None, top_k=3):
        """
        user_needs: Có thể truyền vào 1 chuỗi (String) hoặc 1 danh sách (List of Strings)
        """
        # 0. Chuẩn hóa đầu vào thành List để dễ xử lý
        if isinstance(user_needs, str):
            user_needs = [user_needs]
            
        # Lọc ra các nhu cầu hợp lệ có trong hệ thống
        valid_needs = [n for n in user_needs if n in self.mapping_matrix]
        if not valid_needs:
            return "Lỗi: Không tìm thấy nhu cầu hợp lệ nào."

        # BƯỚC 1: TÍNH TRUNG BÌNH TRỌNG SỐ CHO ĐA NHU CẦU
        combined_weights = {'perf': 0, 'cam': 0, 'batt': 0, 'disp': 0}
        
        for need in valid_needs:
            weights = self.mapping_matrix[need]['weights']
            for key in combined_weights:
                combined_weights[key] += weights.get(key, 0)
                
        # Chia trung bình để tổng trọng số không bị phình to
        num_needs = len(valid_needs)
        user_vector = np.array([
            combined_weights['perf'] / num_needs,
            combined_weights['cam'] / num_needs,
            combined_weights['batt'] / num_needs,
            combined_weights['disp'] / num_needs
        ])

        filtered_df = self.df.copy()

        # BƯỚC 2: GỘP LỌC CỨNG (Vòng lặp AND Logic)
        if max_budget:
            filtered_df = filtered_df[filtered_df['price_vnd'] <= max_budget]

        # Quét qua từng nhu cầu, máy nào không thỏa mãn luật cứng sẽ bị rớt đài dần
        for need in valid_needs:
            hard_filters = self.mapping_matrix[need].get('hard_filters', {})
            for col, condition in hard_filters.items():
                if col in filtered_df.columns:
                    filtered_df = filtered_df[condition(filtered_df[col])]

        # Nếu lọc cứng chọi nhau (Ví dụ: Vừa đòi Flagship vừa đòi Giá rẻ)
        if filtered_df.empty:
            return "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn."

        # BƯỚC 3: CHẤM ĐIỂM BẰNG MA TRẬN
        item_matrix = filtered_df[self.score_features].values
        filtered_df['total_score'] = np.dot(item_matrix, user_vector)

        # Xếp hạng
        final_result = filtered_df.sort_values(by='total_score', ascending=False).head(top_k)
        
        display_cols = ['product_name', 'price_vnd'] + self.score_features + ['total_score']
        return final_result[display_cols].round(2)

# Test

In [50]:
engine = KBRSEngine(df_ready_for_kbrs, MAPPING_MATRIX_CONFIG)

In [51]:
# ==========================================
# 4. CHẠY TEST CÁC KỊCH BẢN THỰC TẾ

print("==== KỊCH BẢN 1: KHÁCH HÀNG QUAY VLOG, NGÂN SÁCH DƯỚI 15 TRIỆU ====")
display(engine.recommend(user_needs='Chup_Hinh_Quay_Phim', max_budget=15000000))
print("\n")

print("==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====")
# Không có điều kiện cứng, ưu tiên pin (0.7), các điểm khác thấp
display(engine.recommend(user_needs='Lien_Lac_Co_Ban', max_budget=4000000))
print("\n")

print("==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====")
# Ép price_segment == 'Flagship', trọng số cân bằng
display(engine.recommend(user_needs='Flagship'))

==== KỊCH BẢN 1: KHÁCH HÀNG QUAY VLOG, NGÂN SÁCH DƯỚI 15 TRIỆU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
108,Xiaomi Redmi Note 13 Pro Plus 5G 8GB 256GB,6490000,3.64,6.18,7.93,6.29,5.99
80,Redmi Note 13 Pro 5G 12GB 512GB,6990000,4.59,6.18,5.59,6.29,5.90
43,Xiaomi Redmi Note 15 Pro 5G 12GB 256GB,10890000,4.44,6.11,5.51,6.29,5.83




==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
365,Xiaomi Redmi Note 13 6GB 128GB,3790000,2.41,3.67,3.98,6.29,4.02
148,Itel RS4 12GB 256GB NFC,3390000,3.98,1.76,4.52,2.29,3.97
349,Itel RS4 8GB 256GB NFC,3290000,3.12,1.76,4.52,2.29,3.88




==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
165,OPPO Find N6 16GB 512GB,63990000,8.49,7.03,6.74,8.29,7.82
252,OPPO Find X9 Ultra 12GB 512GB,48990000,7.63,9.03,6.03,6.29,7.49
257,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,6.88


In [52]:
print("==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====")
# Hệ thống sẽ yêu cầu máy đó phải có: is_heavy_gaming == True VÀ battery_beast == True
display(engine.recommend(user_needs=['Choi_Game', 'Pin_Trau_Sac_Nhanh']))

print("==== MÁY NHẸ, FLAGSHIP ====")
display(engine.recommend(user_needs=['Nho_Gon_Nhe_Tay','Flagship']))


print("\n==== TÌM MÁY CHƠI GAME NHƯNG PHẢI QUAY VLOG MƯỢT (NGÂN SÁCH 35TR) ====")
display(engine.recommend(user_needs=['Choi_Game', 'Chup_Hinh_Quay_Phim'], max_budget=35000000))


print("\n==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====")
# Khách hàng đòi Flagship cao cấp nhưng chọn nhu cầu Giá Rẻ
display(engine.recommend(user_needs=['Flagship', 'Gia_Re'])) 
# Kết quả sẽ in ra: "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI..."

==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
257,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,7.69
165,OPPO Find N6 16GB 512GB,63990000,8.49,7.03,6.74,8.29,7.62
190,OPPO Find X9 16GB 512GB,25990000,8.44,4.41,7.38,6.29,7.47


==== MÁY NHẸ, FLAGSHIP ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
252,OPPO Find X9 Ultra 12GB 512GB,48990000,7.63,9.03,6.03,6.29,7.41
3,Samsung Galaxy S20,21490000,3.51,2.91,2.99,4.29,3.49
362,iPhone 15 256GB | Chính hãng VN/A,20790000,4.86,3.12,2.12,2.00,3.07



==== TÌM MÁY CHƠI GAME NHƯNG PHẢI QUAY VLOG MƯỢT (NGÂN SÁCH 35TR) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
257,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,7.16
269,Xiaomi 15 Ultra 5G 16GB 1TB,28490000,8.68,5.24,6.82,6.29,7.03
308,Xiaomi 15 Ultra 5G 16GB 512GB,26490000,8.18,5.24,6.82,6.29,6.82



==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====


'Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn.'